# Phase 4.1 - Aggressive Model Compression

**Author:** Rafael  **Dataset:** Tiny-ImageNet-200 (200 classes, 64x64 RGB)

Push the best Phase 1-3 models below INT8. Four PyTorch-native quantization
strategies, evaluated on the same axis so bit-widths are directly comparable:

| Method | Bits | Mode | Models |
|---|---|---|---|
| INT8 | W8/A8 | PTQ (calibrate) | all 5 (anchor) |
| INT4 PTQ | W4/A8 | PTQ (calibrate) | all 5 |
| INT4 QAT | W4/A8 | retrain | 3 tiny |
| Mixed | W4-8/A8 per-layer | retrain | 3 tiny |

INT4/mixed use **FakeQuantize simulation** (fbgemm has no INT4 kernel): accuracy is
faithful, size is the theoretical packed size (`theoretical_size_mb`). All eval on GPU.

**Targets:** `mobilenetv2`, `alexnet_bottleneck`, `alexnet_fire`,
`alexnet_depthwisesep` (QAT-unstable edge case), `alexnet_final_fire_residual`.

Layout: 1. Imports 2. Data 3. FP32 baselines 4. Compression helpers
5. INT8 6. INT4 PTQ 7. INT4 QAT 8. Mixed precision 9. Comparison & summaries

## 1. Imports & reproducibility

In [ ]:
import json
import os
import random
import sys
import time
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchinfo
import wandb

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from ml import (
    DataConfig, TrainerConfig, QATConfig,
    create_imagenet_loaders,
    Trainer, make_qat_callback, auto_resume_path,
    load_best_model, build_comparison_table, create_results_summary,
    disk_mb, compute_flops,
    make_qconfig, prepare_sim, calibrate,
    compute_layer_sensitivity, assign_mixed_precision, theoretical_size_mb,
)
from ml import find_fuse_groups
from configs.loader import load_config

from models import (
    MobileNetV2TV, AlexNetBottleneck, AlexNetFire,
    AlexNetDepthwiseSep, AlexNetFinalFireResidual,
)

torch.backends.quantized.engine = "fbgemm"

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR    = project_root / "checkpoints" / "compression_phase4_1"
RESULTS_DIR = project_root / "results" / "compression_phase4_1"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

data_cfg = DataConfig(**load_config("data.yaml"))
data_cfg.seed = SEED
fp32_cfg = TrainerConfig(**load_config("training.yaml"))
comp_cfg = load_config("compression.yaml")
print(device)

## 2. Dataset & loaders

Standard train/val loaders plus a small **calibration loader** (single-worker, no shuffle) used for PTQ range collection and sensitivity analysis.

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")
data_cfg.dataset_path = train_path

train_ds, val_ds, train_loader, val_loader = create_imagenet_loaders(data_cfg)

calib_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=data_cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True,
)

print(f"Train {len(train_ds):,} | Val {len(val_ds):,} | classes {data_cfg.num_classes}")
print(f"Batches: train={len(train_loader)} val={len(val_loader)}")

## 3. Target models & FP32 baselines

The 5 targets live in three different Phase 1-3 checkpoint dirs. We load each
FP32 `*_best.pth`, evaluate it, and record its baseline (accuracy, disk size, params, MACs).

In [ ]:
TARGETS = {
    "mobilenetv2":                 dict(ctor=MobileNetV2TV,            src="baselines_qat_phase1"),
    "alexnet_bottleneck":          dict(ctor=AlexNetBottleneck,        src="compensation_phase3"),
    "alexnet_fire":                dict(ctor=AlexNetFire,              src="compensation_phase3"),
    "alexnet_depthwisesep":        dict(ctor=AlexNetDepthwiseSep,      src="compensation_phase3"),
    "alexnet_final_fire_residual": dict(ctor=AlexNetFinalFireResidual, src="final_architecture_phase4"),
}
ALL_MODELS  = list(TARGETS)
TINY_MODELS = ["alexnet_bottleneck", "alexnet_fire", "alexnet_depthwisesep"]

IMG = data_cfg.img_size
INPUT_SIZE = (1, 3, IMG, IMG)


def load_fp32(name):
    """Load the best FP32 checkpoint for a target from its source phase dir."""
    spec = TARGETS[name]
    src_dir = project_root / "checkpoints" / spec["src"]
    return load_best_model(name, spec["ctor"], src_dir, device, eval_mode=True)


def eval_on(model, loader):
    """Top-1/Top-5/loss via Trainer.evaluate (device follows model)."""
    t = Trainer(model, train_loader, loader, replace(fp32_cfg, use_amp=False),
                next(model.parameters()).device, SAVE_DIR, "eval",
                num_classes=data_cfg.num_classes)
    return t.evaluate(topk=(1, 5))

In [ ]:
fp32_baseline = {}
for name in ALL_MODELS:
    m = load_fp32(name)
    info = torchinfo.summary(m, input_size=INPUT_SIZE, verbose=0)
    ev = eval_on(m, val_loader)
    flops = compute_flops(m.cpu().eval(), input_size=INPUT_SIZE)
    src_dir = project_root / "checkpoints" / TARGETS[name]["src"]
    fp32_baseline[name] = {
        "top1": ev["top1"], "top5": ev["top5"],
        "params_m": info.total_params / 1e6,
        "disk_mb": disk_mb(src_dir / f"{name}_best.pth"),
        "size_mb_32": theoretical_size_mb(m, w_bits=32),
        "macs": flops["macs"],
    }
    print(f"{name:30s} top1={ev['top1']:.2f}% top5={ev['top5']:.2f}% "
          f"params={info.total_params/1e6:.2f}M size={fp32_baseline[name]['disk_mb']:.1f}MB")
    del m; torch.cuda.empty_cache()

## 4. Compression helpers

`fuse_map_for` auto-detects Conv-BN(-ReLU) groups. `run_ptq` calibrates a
fake-quant model (no retraining). `run_qat` fine-tunes one. Every result is
appended to `ROWS` in the shared schema and mirrored to a per-model record.

In [ ]:
ROWS = []
per_model = {n: {"fp32_baseline": fp32_baseline[n], "methods": []} for n in ALL_MODELS}


def fuse_map_for(name):
    """Best-effort auto fuse map (find_fuse_groups recurses into .features etc.)."""
    return find_fuse_groups(TARGETS[name]["ctor"]())


def _load_best_into(model, run_name):
    ckpt = torch.load(SAVE_DIR / f"{run_name}_best.pth", map_location=str(device), weights_only=False)
    model.load_state_dict(ckpt.get("model_state_dict", ckpt))
    return model


def record(name, method, precision, ev, size_mb, bench, notes,
           train_hrs=None, calib_min=None, bits_map=None):
    base = fp32_baseline[name]
    ratio = base["size_mb_32"] / size_mb if size_mb else None
    row = {
        "model": name, "method": method, "precision": precision,
        "fp32_baseline_acc": base["top1"], "compressed_top1_acc": ev["top1"],
        "top5_acc": ev["top5"], "acc_drop_pp": base["top1"] - ev["top1"],
        "compression_ratio": ratio,
        "fp32_size_mb": base["size_mb_32"], "compressed_size_mb": size_mb,
        "latency_ms": bench["latency_ms_per_image"],
        "throughput_img_s": bench["throughput_img_per_s"],
        "params_m": base["params_m"], "acc_per_mb": ev["top1"] / size_mb if size_mb else None,
        "training_time_hrs": train_hrs, "calibration_time_min": calib_min,
        "top1_%": ev["top1"], "top5_%": ev["top5"], "size_MB": size_mb,
        "notes": notes,
    }
    ROWS.append(row)
    rec = dict(row); rec["bits_map"] = bits_map
    per_model[name]["methods"].append(rec)
    print(f"  -> {method:12s} top1={ev['top1']:.2f}% ({row['acc_drop_pp']:+.2f}pp) "
          f"size={size_mb:.2f}MB ratio={ratio:.1f}x")


def wandb_run(name, method, cfg_extra):
    return wandb.init(
        project="alexnet-compression", group="phase-4-1-compression",
        name=f"{name}_{method}",
        config={"arch": name, "method": method, "phase": "4.1",
                "dataset": "tiny-imagenet-200", **cfg_extra},
        tags=["phase-4-1", method, name, "tiny-imagenet", "compression"],
        mode="offline", reinit=True,
    )

In [ ]:
def run_ptq(name, method, w_bits, a_bits, n_calib, precision, notes):
    """Post-training: fuse -> fake-quant -> calibrate -> eval. No retraining."""
    model = load_fp32(name)
    qconfig = make_qconfig(w_bits=w_bits, a_bits=a_bits, per_channel=True)
    sim = prepare_sim(model, fuse_map_for(name), qconfig, device)
    t0 = time.perf_counter()
    sim = calibrate(sim, calib_loader, device, n_samples=n_calib)
    calib_min = (time.perf_counter() - t0) / 60
    ev = eval_on(sim, val_loader)
    bench = Trainer(sim, train_loader, val_loader, replace(fp32_cfg, use_amp=False),
                    device, SAVE_DIR, "bench", num_classes=data_cfg.num_classes).benchmark(warmup=50)
    size_mb = theoretical_size_mb(model, w_bits=w_bits)
    run = wandb_run(name, method, {"w_bits": w_bits, "a_bits": a_bits, "calib_samples": n_calib})
    run.summary.update({"top1": ev["top1"], "top5": ev["top5"], "size_mb": size_mb})
    wandb.finish()
    record(name, method, precision, ev, size_mb, bench, notes, calib_min=calib_min)
    del model, sim; torch.cuda.empty_cache()


def run_qat(name, method, w_bits, a_bits, mcfg, precision, notes, bits_map=None, sim=None):
    """QAT: fine-tune a fake-quant (or mixed) model, reload best, eval."""
    model = load_fp32(name)
    if sim is None:
        qconfig = make_qconfig(w_bits=w_bits, a_bits=a_bits, per_channel=mcfg.get("per_channel", True))
        sim = prepare_sim(model, fuse_map_for(name), qconfig, device)
    run_name = f"{method}_{name}"
    qat_cfg = replace(fp32_cfg, epochs=mcfg["epochs"], lr=mcfg["lr"],
                      weight_decay=mcfg["weight_decay"], use_amp=False)
    cb = make_qat_callback(mcfg["freeze_bn_epoch"], mcfg["disable_observer_epoch"])
    resume = auto_resume_path(SAVE_DIR, run_name)

    if (SAVE_DIR / f"{run_name}_best.pth").exists() and not resume:
        print(f"  (reusing {run_name}_best.pth)")
    else:
        run = wandb_run(name, method, {"w_bits": w_bits, "a_bits": a_bits, **mcfg})
        tr = Trainer(sim, train_loader, val_loader, qat_cfg, device, SAVE_DIR, run_name,
                     num_classes=data_cfg.num_classes, wandb_run=run, epoch_callback=cb,
                     log_file=SAVE_DIR / f"{run_name}.log")
        fit = tr.fit(resume_from=resume)
        wandb.finish()
        per_model[name].setdefault("train_time_hrs", {})[method] = fit["total_training_time_s"] / 3600

    sim = _load_best_into(sim, run_name).eval()
    ev = eval_on(sim, val_loader)
    bench = Trainer(sim, train_loader, val_loader, qat_cfg, device, SAVE_DIR, "bench",
                    num_classes=data_cfg.num_classes).benchmark(warmup=50)
    size_mb = theoretical_size_mb(model, w_bits=w_bits, bits_map=bits_map)
    train_hrs = per_model[name].get("train_time_hrs", {}).get(method)
    record(name, method, precision, ev, size_mb, bench, notes,
           train_hrs=train_hrs, bits_map=bits_map)
    del model, sim; torch.cuda.empty_cache()

## 5. INT8 (anchor)

Uniform fake-quant W8/A8 PTQ across all 5 models - the known-good reference point. (Real INT8-QAT numbers from Phases 1-3 remain valid cross-checks.)

In [ ]:
for name in ALL_MODELS:
    print(name)
    run_ptq(name, "int8", w_bits=8, a_bits=8,
            n_calib=comp_cfg["int4_ptq"]["calibration_samples"],
            precision="int8", notes="fake-quant W8/A8 PTQ")

## 6. INT4 PTQ

Weights to 4-bit post-training (per-channel), activations INT8. The fast low-bit baseline.

In [ ]:
for name in ALL_MODELS:
    print(name)
    run_ptq(name, "int4_ptq", w_bits=4, a_bits=8,
            n_calib=comp_cfg["int4_ptq"]["calibration_samples"],
            precision="int4", notes="fake-quant W4/A8 PTQ, per-channel")

## 7. INT4 QAT

Retrain under 4-bit weight fake-quant on the 3 tiny models - does training recover the PTQ gap? (Watch **alexnet_depthwisesep**, the -2.92 pp QAT edge case.)

In [ ]:
mcfg_int4 = comp_cfg["int4_qat"]
for name in TINY_MODELS:
    print(name)
    run_qat(name, "int4_qat", w_bits=4, a_bits=8, mcfg=mcfg_int4,
            precision="int4", notes="fake-quant W4/A8 QAT, per-channel")

## 8. Mixed precision (INT4/INT8 per-layer)

Rank layers by INT4 sensitivity, keep the top `int8_ratio` at INT8 and the rest at INT4, then fine-tune. Tests whether per-layer mixing beats uniform INT4.

In [ ]:
import copy as _copy
mcfg_mix = comp_cfg["mixed_precision"]
for name in TINY_MODELS:
    print(name)
    model = load_fp32(name)
    base = _copy.deepcopy(model).to(device).train()
    # sensitivity + per-layer qconfig on the UNFUSED graph so layer names match,
    # THEN fuse (carries each conv's qconfig into the fused module) and prepare_qat.
    sens = compute_layer_sensitivity(base, calib_loader, device, w_bits=4)
    bits_map = assign_mixed_precision(base, sens, int8_ratio=mcfg_mix["int8_ratio"],
                                      per_channel=mcfg_mix["per_channel"])
    base.qconfig = make_qconfig(8, 8, mcfg_mix["per_channel"])  # default for unmarked mods
    fm = fuse_map_for(name)
    if fm:
        try:
            torch.ao.quantization.fuse_modules_qat(base, fm, inplace=True)
        except Exception as e:
            print("  fusion skipped:", e)
    base.train()  # fuse_modules_qat flips to eval; prepare_qat requires train mode
    torch.ao.quantization.prepare_qat(base, inplace=True)
    n8 = sum(v == 8 for v in bits_map.values()); n4 = sum(v == 4 for v in bits_map.values())
    print(f"  layers: INT8={n8} INT4={n4}")
    run_qat(name, "mixed", w_bits=4, a_bits=8, mcfg=mcfg_mix,
            precision="int4/8", notes=f"per-layer INT4/INT8 ({n8} hi / {n4} lo)",
            bits_map=bits_map, sim=base)
    del model, base; torch.cuda.empty_cache()

## 9. Comparison table, per-model summaries & experiment summary

In [ ]:
df = build_comparison_table(ROWS)
cols = ["model","method","precision","fp32_baseline_acc","compressed_top1_acc","acc_drop_pp",
        "compression_ratio","compressed_size_mb","latency_ms","acc_per_mb","notes"]
df_out = df[[c for c in cols if c in df.columns]].copy()
df.to_csv(RESULTS_DIR / "final_comparison.csv", index=False)

by_method = (df.groupby("method")
               .agg(mean_top1=("compressed_top1_acc","mean"),
                    mean_drop_pp=("acc_drop_pp","mean"),
                    mean_ratio=("compression_ratio","mean"),
                    mean_acc_per_mb=("acc_per_mb","mean"),
                    n=("model","count"))
               .reset_index())
by_method.to_csv(RESULTS_DIR / "compression_by_method.csv", index=False)

for name, rec in per_model.items():
    with open(RESULTS_DIR / f"{name}_compression_summary.json", "w") as f:
        json.dump(rec, f, indent=2, default=str)

df_out

In [ ]:
def dominated(r, others):
    for o in others:
        if (o["compressed_top1_acc"] >= r["compressed_top1_acc"]
                and o["compressed_size_mb"] <= r["compressed_size_mb"]
                and o["latency_ms"] <= r["latency_ms"]
                and (o["compressed_top1_acc"] > r["compressed_top1_acc"]
                     or o["compressed_size_mb"] < r["compressed_size_mb"]
                     or o["latency_ms"] < r["latency_ms"])):
            return True
    return False

recs = df.to_dict("records")
pareto = [r for r in recs if not dominated(r, recs)]
pareto_df = pd.DataFrame(pareto).sort_values("compressed_top1_acc", ascending=False)
pareto_df[[c for c in cols if c in pareto_df.columns]].to_csv(RESULTS_DIR / "pareto_frontier.csv", index=False)
print(f"Pareto-optimal: {len(pareto)} of {len(recs)}")
pareto_df[["model","method","compressed_top1_acc","compressed_size_mb","latency_ms"]]

In [ ]:
create_results_summary(
    results={
        "phase": "4.1", "title": "Aggressive Model Compression",
        "models": ALL_MODELS,
        "methods": ["int8", "int4_ptq", "int4_qat", "mixed"],
        "fp32_baseline": fp32_baseline,
        "rows": ROWS,
        "by_method": by_method.to_dict("records"),
        "pareto": pareto_df[["model","method","compressed_top1_acc","compressed_size_mb"]].to_dict("records"),
    },
    config=fp32_cfg,
    output_path=RESULTS_DIR / "experiment_summary.json",
)
print("Saved:", RESULTS_DIR / "experiment_summary.json")

## W&B - syncing offline runs

All runs logged offline under project **`alexnet-compression`**, group
**`phase-4-1-compression`**, phase tag `4.1`. Sync when ready:

```bash
wandb sync --sync-all
```